# QwenPRM Wrapper — Score Inspection

Inspect the `QwenPRM` wrapper class (`reward_models.py`) through
the shared `score(questions, answers, batch_size=...)` interface.

Seven examples: (1) the flamingo problem — model-card sanity
check; (2) a batched call — two questions, mixed answer counts;
(3–4) a preamble scored as Step 0 (toy, then verbatim generated
output — not penalised in either); (5–6) trailing `"\n\n"`
non-terminal candidates — the `PRM._split_steps` fix; (7) the
pre-fix behavior reproduced via a temporary monkey-patch: the
bogus extra step scores like a holistic trajectory-level
P(correct), which `agg_strategy="last"` then silently returned
in place of the true last-step score.

Note: the model card specifies `bfloat16`, but the V100 (sm_70)
has no bf16 support, so we load `float16`. fp16 preserves step
*rankings* but can drift absolute scores slightly.

Env: runs under `py311` (transformers 4.57). The bundled remote
code (`modeling_qwen2_rm.py`) calls a cache API removed in newer
transformers; `QwenPRM` uses `use_cache=False` to sidestep it.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc
import sys
sys.path.append("..")

import torch

from notebook_utils import gpu_mem_used_gb, print_step_scores
from core.reward_models import QwenPRM

In [2]:
# Model paths
base_dir = "/groups/chichengz/tnn/datasets"
prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

## Load the PRM

In [3]:
# fp16 for V100 (sm_70); model card recommends bf16 (Ampere+),
# so absolute scores may drift slightly on Ampere GPUs.
prm = QwenPRM(prm_dir)

print(f"sep token id  : {prm.sep_token_id}")
print(f"dtype         : {next(prm.model.parameters()).dtype}")
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

sep token id  : 151651
dtype         : torch.float16
GPU memory used: 13.87 GB


## Example 1 — flamingo problem (single question, single answer)

Model-card reference scores (bf16): `[1.0, 0.1904, 0.9766, 1.0]`.
On V100/fp16 expect close, not exact.

In [4]:
# Toy example from the Qwen2.5-Math-PRM-7B model card.
# Double-backslash LaTeX so "\t"/"\b" don't become control chars.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [5]:
# One question, one candidate answer whose steps are joined by
# "\n\n". score() returns [question][answer][step].
scores = prm.score([problem], [["\n\n".join(reasoning_steps)]])

print("=== Flamingo trajectory ===")
print_step_scores(reasoning_steps, scores[0][0])

=== Flamingo trajectory ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1580
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...


## Example 2 — batched call (two questions, mixed answer counts)

One `prm.score()` call with two questions: the flamingo problem
(one answer) and the algebra problem (two candidate answers —
correct and wrong). The base class flattens all pairs, scores in
one batched forward pass, and reshapes back to
`[question][answer][step]`.

In [6]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [7]:
# Two questions; flamingo has 1 answer, algebra has 2.
# score() flattens to 3 pairs, batches, reshapes back.
questions = [problem, algebra_problem]
answers = [
    ["\n\n".join(reasoning_steps)],
    ["\n\n".join(correct_steps), "\n\n".join(wrong_steps)],
]

batch_scores = prm.score(questions, answers, batch_size=4)

print("=== Flamingo (Q0, A0) ===")
print_step_scores(reasoning_steps, batch_scores[0][0])

print("\n=== Algebra correct (Q1, A0) ===")
print_step_scores(correct_steps, batch_scores[1][0])

print("\n=== Algebra wrong — step 2 divides by 2 (Q1, A1) ===")
print_step_scores(wrong_steps, batch_scores[1][1])

=== Flamingo (Q0, A0) ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1581
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...

=== Algebra correct (Q1, A0) ===
Step 1: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).

=== Algebra wrong — step 2 divides by 2 (Q1, A1) ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0102
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.5127
Therefore, the final answer is \boxed{6}.


## Example 3 — preamble as Step 0

Qwen-Math-Instruct opens a *fresh* generation with a lead-in
sentence ("To find the angle…", "To solve this…") instead of
jumping to a labelled first step — see
`examine_llm_generation_templates_qwen_v1`. In the search tree
this preamble is the **root step**. Question: does the PRM
penalise it?

Here we score two trajectories in one call: a control — the
algebra *correct* trajectory from Example 2, relabelled
`"## Step N:"` — and the same steps with the preamble prepended
as **Step 0**. Two things to read off:

1. **Step 0 score** — does the PRM mark a content-free lead-in as
   low-reward (a "bad step")?
2. **Steps 1–4 vs the control** — the two trajectories' Steps 1–4
   are byte-identical, so any score shift is attributable to the
   preamble alone.

Same `prm.score()` interface; the preamble is just another
`\n\n`-separated step, so it gets its own `<extra_0>` position.

**Observed: no penalty.** The content-free preamble scores 0.9902
— high, not treated as a bad step. And it barely moves the
downstream steps: Steps 1–4 shift by ≤0.001 vs the control
(0.9995→0.9985 on Step 1, the rest identical).

In [8]:
# Control: "## Step N:"-labelled working steps, no preamble — same
# underlying content as correct_steps (Example 2), just labelled.
labelled_steps = [
    "## Step 1: We need solve the equation 3x + 5 = 17.",
    "## Step 2: Subtracting 5 from both sides gives 3x = 12.",
    "## Step 3: Dividing both sides by 3 gives x = 4.",
    "## Step 4: Therefore, the answer is (\\boxed{4}).",
]

# Same labelled_steps with a content-free preamble prepended as
# Step 0 — the lead-in Qwen-Math emits on a fresh turn. Steps 1-4
# are BYTE-IDENTICAL to labelled_steps, so preamble_steps vs
# labelled_steps isolates exactly one variable: does prepending the
# preamble change the (otherwise identical) Step 1-4 scores?
preamble = "To solve this problem, we will follow these steps:"
preamble_steps = [preamble, *labelled_steps]

ex3_scores = prm.score(
    [algebra_problem, algebra_problem],
    [["\n\n".join(preamble_steps)], ["\n\n".join(labelled_steps)]],
)

print("=== Algebra correct, '## Step N:'-labelled (control) ===")
print_step_scores(labelled_steps, ex3_scores[1][0])

print("\n=== Algebra correct WITH preamble as Step 0 ===")
print_step_scores(preamble_steps, ex3_scores[0][0], start=0)


=== Algebra correct, '## Step N:'-labelled (control) ===
Step 1: P(correct) = 0.9995
## Step 1: We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9990
## Step 2: Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 0.9980
## Step 3: Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
## Step 4: Therefore, the answer is (\boxed{4}).

=== Algebra correct WITH preamble as Step 0 ===
Step 0: P(correct) = 0.9902
To solve this problem, we will follow these steps:
Step 1: P(correct) = 0.9985
## Step 1: We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9990
## Step 2: Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 0.9980
## Step 3: Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
## Step 4: Therefore, the answer is (\boxed{4}).


## Example 4 — real generated preamble (verbatim model output)

Example 3 used a hand-written preamble on a clean trajectory.
Here we score the **verbatim** Qwen2.5-Math-7B-Instruct output
from `examine_llm_generation_templates_qwen_v1` (native template,
angle-between-lines question): the native/fresh preamble as
**Step 0**, then the real Step 1/2 (continuation prefix) and the
generated Step 3.

The trajectory is **incomplete** — it stops at the dot product,
has no `\boxed{}`, and the partial work doesn't reach the true
answer (90°). That's fine: we only compare the per-step score of
the preamble (Step 0) against the labelled `## Step N` steps in
the same trajectory, not final-answer correctness. This is the
realistic version of Example 3's question — does the PRM penalise
the model's *actual* root output?

**Observed: no penalty here either.** The real preamble scores
0.9990, consistent with Example 3: preambles as root steps are
not what the PRM punishes. The low score lands on Step 1
(0.0598), and deservedly — Step 1 reads the coefficients off
`2x = 3y = -z` as the direction vector `(2, 3, -1)`, but the
actual direction is `(1/2, 1/3, -1) ∝ (3, 2, -6)`. The PRM
correctly flags the real error and lets the content-free
lead-in pass.

In [9]:
# Verbatim Qwen2.5-Math-7B-Instruct output from
# examine_llm_generation_templates_qwen_v1 (native template).
gen_question = (
    "The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]"
    "is a line.\n\nThe set of points $(x,y,z)$ that satisfy\n"
    "\\[6x = -y = -4z\\]is another line.\n\nFind the angle "
    "between these lines, in degrees."
)

# Step 0 = the native/fresh preamble the model emitted; Steps 1-2 =
# the continuation prefix; Step 3 = the generated continuation.
gen_steps = [
    "To find the angle between the two lines given by the "
    "equations \\(2x = 3y = -z\\) and \\(6x = -y = -4z\\), we "
    "first need to determine the direction vectors of these "
    "lines.",

    "## Step 1: Identify the direction vectors of the lines.\n"
    "For the first line the direction vector is (2, 3, -1); for "
    "the second line it is (6, -1, -4).",

    "## Step 2: Recall the formula for the angle between "
    "vectors.\ncos(theta) = (a . b) / (|a| |b|).",

    "## Step 3: Compute the dot product of the direction "
    "vectors.\na . b = 2*6 + 3*(-1) + (-1)*(-4) = 12 - 3 + 4 = 13.",
]

gen_scores = prm.score([gen_question], [["\n\n".join(gen_steps)]])

print("=== Generated trajectory, preamble as Step 0 ===")
print_step_scores(gen_steps, gen_scores[0][0], start=0)

=== Generated trajectory, preamble as Step 0 ===
Step 0: P(correct) = 0.9990
To find the angle between the two lines given by the equatio...
Step 1: P(correct) = 0.0598
## Step 1: Identify the direction vectors of the lines.
For ...
Step 2: P(correct) = 0.9609
## Step 2: Recall the formula for the angle between vectors....
Step 3: P(correct) = 0.9526
## Step 3: Compute the dot product of the direction vectors....


## Example 5 — trailing "\n\n" (toy, simulates a non-terminal MCTS candidate)

vLLM's `include_stop_str_in_output=True` with `stop=["\n\n"]` means a
**non-terminal** candidate's text keeps the `"\n\n"` stop string that
cut generation short — only EOS/length-terminated candidates lack it.
Before the `PRM._split_steps` fix, `answer.split("\n\n")` on such a
trailing separator produced a bogus empty extra "step" (its own scored
`<extra_0>` position, silently corrupting `agg_strategy="last"`); with
the fix (`removesuffix("\n\n")` before splitting) this no longer
happens.

Toy check: append a trailing `"\n\n"` to `labelled_steps` (as if the
candidate were still mid-search) and score it. We expect exactly 4
scores — matching `labelled_steps`, not 5.


In [10]:
# Same labelled_steps as Example 3, but joined with a trailing
# "\n\n" -- the shape of a non-terminal MCTS candidate (stop string
# kept via include_stop_str_in_output=True).
trailing_answer = "\n\n".join(labelled_steps) + "\n\n"

# What a naive `answer.split("\n\n")` (the pre-fix behavior) would
# have produced: a bogus trailing empty "step".
naive_split = trailing_answer.split("\n\n")
print(f"naive split('\\\\n\\\\n') -> {len(naive_split)} pieces, "
      f"last piece = {naive_split[-1]!r}")

trailing_scores = prm.score([algebra_problem], [[trailing_answer]])

print(f"\nprm.score() -> {len(trailing_scores[0][0])} scores "
      f"(expected {len(labelled_steps)}, matching labelled_steps)")
print("=== Trailing '\\n\\n' candidate (toy) ===")
print_step_scores(labelled_steps, trailing_scores[0][0])


naive split('\\n\\n') -> 5 pieces, last piece = ''



prm.score() -> 4 scores (expected 4, matching labelled_steps)
=== Trailing '\n\n' candidate (toy) ===
Step 1: P(correct) = 0.9995
## Step 1: We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9990
## Step 2: Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 0.9980
## Step 3: Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
## Step 4: Therefore, the answer is (\boxed{4}).


## Example 6 — trailing "\n\n" (real, non-terminal generation)

Same check on the verbatim Example 4 trajectory (`gen_steps`), but now
treated as **non-terminal** — the vLLM stop string that cut the last
generated step (`## Step 3: ...`) is still attached, exactly as
`_generate_candidates` would hand it to `prm.score()` mid-search
(before any call-site `removesuffix` cleanup, which the three MCTS
launchers apply only to protect their own embedding path — the PRM
itself must handle this on its own via `_split_steps`).

We expect exactly 4 scores — matching `gen_steps`, not 5 — with Step 0
still the preamble.


In [11]:
# Same verbatim trajectory as Example 4, but with the trailing
# "\n\n" stop string still attached (i.e., as a live, non-terminal
# MCTS candidate would appear before any call-site cleanup).
gen_trailing_answer = "\n\n".join(gen_steps) + "\n\n"

naive_split = gen_trailing_answer.split("\n\n")
print(f"naive split('\\\\n\\\\n') -> {len(naive_split)} pieces, "
      f"last piece = {naive_split[-1]!r}")

gen_trailing_scores = prm.score([gen_question], [[gen_trailing_answer]])

print(f"\nprm.score() -> {len(gen_trailing_scores[0][0])} scores "
      f"(expected {len(gen_steps)}, matching gen_steps)")
print("=== Trailing '\\n\\n' candidate (real, non-terminal) ===")
print_step_scores(gen_steps, gen_trailing_scores[0][0], start=0)


naive split('\\n\\n') -> 5 pieces, last piece = ''



prm.score() -> 4 scores (expected 4, matching gen_steps)
=== Trailing '\n\n' candidate (real, non-terminal) ===
Step 0: P(correct) = 0.9990
To find the angle between the two lines given by the equatio...
Step 1: P(correct) = 0.0598
## Step 1: Identify the direction vectors of the lines.
For ...
Step 2: P(correct) = 0.9609
## Step 2: Recall the formula for the angle between vectors....
Step 3: P(correct) = 0.9526
## Step 3: Compute the dot product of the direction vectors....


## Example 7 — pre-fix behavior (temporary monkey-patch of `_split_steps`)

Examples 5-6 confirmed the **current, fixed** `PRM._split_steps`
handles a trailing `"\n\n"` correctly. To see what the **pre-fix**
behavior (`answer.split("\n\n")`, no `removesuffix` first) actually
did in real model inference -- not just at the string level -- we
temporarily monkey-patch `PRM._split_steps` back to the naive split
for a single `prm.score()` call, using `unittest.mock.patch.object`
as a context manager so it's guaranteed to restore afterward
regardless of errors. No source file is touched.

Four candidates, all with the trailing stop string attached (i.e.
non-terminal): the *wrong* algebra trajectory from Example 2 and
the real trajectory from Example 6, each scored twice -- the full
trajectory, and the same trajectory **cut right after its bad
step** (toy: Step 2, the wrong division; real: Step 1, the low
step). In every case the bogus empty "step" gets its own
`<extra_0>` scored position and `agg_strategy="last"` -- the
aggregation this bug affected most -- silently returns that bogus
score instead of the true last-step score.

The cut-after-bad-step candidates are the search-critical case: a
non-terminal MCTS candidate whose last completed step is bad
*should* be valued by that step's low score under `agg="last"` --
does the bogus score mask the bad step (making a bad branch look
healthy mid-search) or track it?

**Observed: it tracks it.** Cut right after the bad step, the
bogus score sits next to the bad step's (toy 0.0115 vs 0.0103;
real 0.0861 vs 0.0593) -- no masking. The divergence appears when
earlier and later steps disagree: the full wrong trajectory's
bogus score (0.30) falls *between* the bad step (0.01) and the
recovered last step (0.51). So the bogus position scores like a
*holistic* trajectory-level P(correct) -- the pre-fix bug
substituted trajectory-level value for last-step value on every
non-terminal candidate: a systematic but correlated distortion,
never making a just-failed branch look healthy.

In [12]:
from unittest.mock import patch

from core.reward_models import PRM
from core.scoring import aggregate_scores

# Pre-fix behavior: split() with no removesuffix() first -- reproduces
# the bug exactly as it behaved before the fix in reward_models.py.
naive_split_steps = staticmethod(lambda answer: answer.split("\n\n"))

# Four non-terminal candidates (trailing stop string attached): the
# full wrong/real trajectories from Examples 5-6, plus the same
# trajectories CUT right after their bad step (toy: Step 2, the
# wrong division; real: Step 1, the low step) -- the search-critical
# case where agg="last" should return that bad step's low score.
wrong_trailing_answer = "\n\n".join(wrong_steps) + "\n\n"
wrong_badstep_answer = "\n\n".join(wrong_steps[:2]) + "\n\n"
gen_badstep_answer = "\n\n".join(gen_steps[:2]) + "\n\n"

with patch.object(PRM, "_split_steps", naive_split_steps):
    toy_prefix_scores = prm.score(
        [algebra_problem],
        [[wrong_trailing_answer, wrong_badstep_answer]],
    )
    real_prefix_scores = prm.score(
        [gen_question],
        [[gen_trailing_answer, gen_badstep_answer]],
    )

# Outside the `with` block, PRM._split_steps is back to the fixed
# version -- confirm the patch didn't leak.
print(f"_split_steps restored to fixed version: "
      f"{PRM._split_steps(trailing_answer) == labelled_steps}")

toy_scores = toy_prefix_scores[0][0]
toy_bad_scores = toy_prefix_scores[0][1]
real_scores = real_prefix_scores[0][0]
real_bad_scores = real_prefix_scores[0][1]

# The exact (buggy) "steps" the naive split produced -- a trailing
# empty string is the bogus extra step.
print("\n=== PRE-FIX: wrong trajectory, full (bogus 4th step) ===")
print_step_scores(wrong_trailing_answer.split("\n\n"), toy_scores)
print(
    f"agg_strategy='last' -> "
    f"{aggregate_scores(toy_scores, 'last'):.4f} "
    f"(bogus step, not the true last step's {toy_scores[-2]:.4f})"
)

print("\n=== PRE-FIX: wrong trajectory CUT after the wrong step "
      "(bogus 3rd step) ===")
print_step_scores(wrong_badstep_answer.split("\n\n"), toy_bad_scores)
print(
    f"agg_strategy='last' -> "
    f"{aggregate_scores(toy_bad_scores, 'last'):.4f} "
    f"(bogus step, not the bad last step's {toy_bad_scores[-2]:.4f})"
)

print("\n=== PRE-FIX: real trajectory, full (bogus 5th step) ===")
print_step_scores(
    gen_trailing_answer.split("\n\n"), real_scores, start=0,
)
print(
    f"agg_strategy='last' -> "
    f"{aggregate_scores(real_scores, 'last'):.4f} "
    f"(bogus step, not the true last step's {real_scores[-2]:.4f})"
)

print("\n=== PRE-FIX: real trajectory CUT after the low step "
      "(bogus 3rd step) ===")
print_step_scores(
    gen_badstep_answer.split("\n\n"), real_bad_scores, start=0,
)
print(
    f"agg_strategy='last' -> "
    f"{aggregate_scores(real_bad_scores, 'last'):.4f} "
    f"(bogus step, not the bad last step's {real_bad_scores[-2]:.4f})"
)

_split_steps restored to fixed version: True

=== PRE-FIX: wrong trajectory, full (bogus 4th step) ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0103
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.5142
Therefore, the final answer is \boxed{6}.
Step 4: P(correct) = 0.3000

agg_strategy='last' -> 0.3000 (bogus step, not the true last step's 0.5142)

=== PRE-FIX: wrong trajectory CUT after the wrong step (bogus 3rd step) ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0103
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.0115

agg_strategy='last' -> 0.0115 (bogus step, not the bad last step's 0.0103)

=== PRE-FIX: real trajectory, full (bogus 5th step) ===
Step 0: P(correct) = 0.9990
To find the angle between the two lines given by the equatio...
Step 1: P(correct) = 0.0593
## Step 1: Identify the direction vectors of the lines.
For ...
Step 2: P(correct) = 0

## Example 8 — agg_strategy comparison (min / prod / last)

`core/scoring.py::aggregate_scores` supports three strategies:
`"min"` (weakest step), `"prod"` (product of all step
P(correct)s), `"last"` (final step only — every other example in
this notebook, and every table in `docs/exp-comparison.md`, fixes
`agg_strategy="last"`). This example scores the two full,
terminal trajectories already in this notebook — the toy *wrong*
algebra answer (Example 2, `wrong_steps`) and the *real* generated
answer (Example 4, `gen_steps`) — under all three, current
(post-fix) `_split_steps` behavior throughout.

Both trajectories contain one clearly bad step (toy: Step 2
divides by 2 instead of 3; real: Step 1's direction vector). The
question: does the choice of aggregation change which trajectory
looks better, or by how much?

**Observed: dramatically, in both cases — more so than
RLHFlowPRM** (see `examine_prm_scores_rlhflowprm_v1`, Example 8).
QwenPRM scores the bad step with much higher confidence (near 0)
than RLHFlowPRM did (~0.24–0.50), so the `last`-vs-`min` gap is
larger here: toy trajectory, Step 2's division error scores
**0.0103**, but the trajectory recovers a plausible closing
sentence (Step 3, 0.5132), so `agg="last"` reports **0.5132** —
**~50× higher** than `agg="min"`. `prod` (0.0052) tracks `min`
(dominated by the near-zero factor). On the real trajectory: Step
1's wrong direction vector scores **0.0598**, but Step 3's
unrelated dot-product computation scores 0.9526, so `last` reports
**0.9526** — **~16× higher** than `min`. `prod`=0.0546, again close
to `min`. Same qualitative story as RLHFlowPRM (`last` is the most
exploitable metric for a trajectory that errs mid-way then
recovers a confident-sounding ending; `min` and `prod` both catch
the real error), but the effect size is larger here because
QwenPRM is more decisive about flagging the bad step in the first
place.

In [13]:
# Score both trajectories fresh (terminal, no trailing "\n\n") under
# the current (fixed) _split_steps -- same scores as Examples 2/4.
toy_full_scores = prm.score(
    [algebra_problem], [["\n\n".join(wrong_steps)]],
)[0][0]
real_full_scores = prm.score(
    [gen_question], [["\n\n".join(gen_steps)]],
)[0][0]

print("=== toy: wrong algebra trajectory ===")
print_step_scores(wrong_steps, toy_full_scores)
for strategy in ("min", "prod", "last"):
    print(
        f"agg_strategy={strategy!r:>6} -> "
        f"{aggregate_scores(toy_full_scores, strategy):.4f}"
    )

print("\n=== real: generated trajectory (preamble as Step 0) ===")
print_step_scores(gen_steps, real_full_scores, start=0)
for strategy in ("min", "prod", "last"):
    print(
        f"agg_strategy={strategy!r:>6} -> "
        f"{aggregate_scores(real_full_scores, strategy):.4f}"
    )

=== toy: wrong algebra trajectory ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0103
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.5132
Therefore, the final answer is \boxed{6}.
agg_strategy= 'min' -> 0.0103
agg_strategy='prod' -> 0.0052
agg_strategy='last' -> 0.5132

=== real: generated trajectory (preamble as Step 0) ===
Step 0: P(correct) = 0.9990
To find the angle between the two lines given by the equatio...
Step 1: P(correct) = 0.0598
## Step 1: Identify the direction vectors of the lines.
For ...
Step 2: P(correct) = 0.9609
## Step 2: Recall the formula for the angle between vectors....
Step 3: P(correct) = 0.9526
## Step 3: Compute the dot product of the direction vectors....
agg_strategy= 'min' -> 0.0598
agg_strategy='prod' -> 0.0546
agg_strategy='last' -> 0.9526


## Cleanup

In [14]:
del prm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

GPU memory used: 13.57 GB
